# Your First Dynamic Factor Model in 2 Minutes

[![Open In Colab](https://colab.research.google.com/assets/colab-badge.svg)](https://colab.research.google.com/github/m-deane/course-creator/blob/main/courses/dynamic-factor-models/quick-starts/00_hello_dfm.ipynb)

**Goal:** Extract hidden factors from multiple correlated time series and see them in action.

**Time:** 2 minutes to working code, 10 minutes total

---

## Quick Win: Working DFM Right Now

Run this cell - you'll extract 2 hidden factors from 4 correlated time series.

In [ ]:
import numpy as np
import pandas as pd
import matplotlib.pyplot as plt
from statsmodels.tsa.statespace.dynamic_factor import DynamicFactor

# Create synthetic data: 4 series driven by 2 hidden factors
np.random.seed(42)
n = 200
factor1 = np.cumsum(np.random.randn(n)) * 0.5  # Trend factor
factor2 = np.sin(np.linspace(0, 4*np.pi, n)) * 2  # Cyclical factor

# 4 observed series = linear combinations of factors + noise
data = pd.DataFrame({
    'Series_1': 1.2*factor1 + 0.8*factor2 + np.random.randn(n)*0.5,
    'Series_2': 0.9*factor1 + 1.1*factor2 + np.random.randn(n)*0.5,
    'Series_3': 1.5*factor1 + 0.3*factor2 + np.random.randn(n)*0.5,
    'Series_4': 0.7*factor1 + 1.3*factor2 + np.random.randn(n)*0.5,
})

# Fit Dynamic Factor Model
model = DynamicFactor(data, k_factors=2, factor_order=2)
results = model.fit(disp=False)

# Extract estimated factors
estimated_factors = results.factors.filtered

print("✓ Model fitted successfully!")
print(f"Log-likelihood: {results.llf:.2f}")
print(f"\nFactor loadings (how much each series depends on each factor):")
print(results.coefficients_of_determination)

## Visualize: What Just Happened?

The model found 2 hidden factors that explain the common movements in your 4 time series.

In [ ]:
fig, axes = plt.subplots(3, 1, figsize=(12, 10))

# Plot 1: Original time series
data.plot(ax=axes[0], alpha=0.7, linewidth=1.5)
axes[0].set_title('Original Time Series (4 correlated series)', fontsize=14, fontweight='bold')
axes[0].set_xlabel('Time')
axes[0].set_ylabel('Value')
axes[0].legend(loc='upper left')
axes[0].grid(True, alpha=0.3)

# Plot 2: Estimated factors
axes[1].plot(estimated_factors[:, 0], label='Factor 1 (Trend)', linewidth=2, color='darkred')
axes[1].plot(estimated_factors[:, 1], label='Factor 2 (Cycle)', linewidth=2, color='darkblue')
axes[1].set_title('Extracted Hidden Factors (2 factors drive all 4 series)', fontsize=14, fontweight='bold')
axes[1].set_xlabel('Time')
axes[1].set_ylabel('Factor Value')
axes[1].legend()
axes[1].grid(True, alpha=0.3)

# Plot 3: True vs Estimated Factor 1 (for comparison)
axes[2].plot(factor1, label='True Factor 1', linewidth=2, alpha=0.6, linestyle='--')
axes[2].plot(estimated_factors[:, 0], label='Estimated Factor 1', linewidth=2)
axes[2].set_title('Model Recovery: True vs Estimated Factor 1', fontsize=14, fontweight='bold')
axes[2].set_xlabel('Time')
axes[2].set_ylabel('Value')
axes[2].legend()
axes[2].grid(True, alpha=0.3)

plt.tight_layout()
plt.show()

print("\n💡 Key Insight: The model discovered the hidden factors without knowing they existed!")

## How It Works

**Dynamic Factor Models assume:**
```
Each observed series = Weighted sum of K hidden factors + Noise
```

**What the model does:**
1. Looks for common patterns across all series
2. Extracts K factors that capture these patterns
3. Each factor evolves over time (hence "dynamic")
4. Each series has different "loadings" on each factor

**Key parameters:**
- `k_factors=2`: Number of hidden factors (you choose this)
- `factor_order=2`: How dynamic each factor is (AR order)

**The magic:** Reduces 4 correlated series → 2 independent factors!

## Modify This: Experiment with Parameters

Try changing these values and see what happens:

In [ ]:
# TODO: Try different values
K_FACTORS = 2        # Try: 1, 2, 3, 4
FACTOR_ORDER = 2     # Try: 0, 1, 2, 4
N_SERIES = 4         # Try: 3, 5, 10

# Generate new data with N_SERIES variables
np.random.seed(42)
n = 200
factor1 = np.cumsum(np.random.randn(n)) * 0.5
factor2 = np.sin(np.linspace(0, 4*np.pi, n)) * 2

data_experiment = pd.DataFrame({
    f'Series_{i+1}': np.random.randn()*factor1 + np.random.randn()*factor2 + np.random.randn(n)*0.5
    for i in range(N_SERIES)
})

# Fit new model
model_exp = DynamicFactor(data_experiment, k_factors=K_FACTORS, factor_order=FACTOR_ORDER)
results_exp = model_exp.fit(disp=False)

print(f"Model with {K_FACTORS} factors, order {FACTOR_ORDER}, {N_SERIES} series")
print(f"Log-likelihood: {results_exp.llf:.2f}")
print(f"AIC: {results_exp.aic:.2f} (lower is better)")

# Quick visualization
fig, ax = plt.subplots(figsize=(12, 4))
factors_exp = results_exp.factors.filtered
for i in range(K_FACTORS):
    ax.plot(factors_exp[:, i], label=f'Factor {i+1}', linewidth=2)
ax.set_title(f'Extracted {K_FACTORS} Factors', fontsize=14, fontweight='bold')
ax.legend()
ax.grid(True, alpha=0.3)
plt.tight_layout()
plt.show()

print("\n🔍 Did you notice?")
print("- More factors = better fit but risk of overfitting")
print("- Higher order = factors can capture more complex dynamics")
print("- More series = more data to learn from")

## Copy-Paste Template: Use in Your Projects

Use this template with your own data:

In [ ]:
from statsmodels.tsa.statespace.dynamic_factor import DynamicFactor
import pandas as pd

# STEP 1: Load your data (rows=time, columns=variables)
# your_data = pd.read_csv('your_data.csv', index_col=0, parse_dates=True)

# STEP 2: Fit Dynamic Factor Model
model = DynamicFactor(
    your_data,           # Your DataFrame
    k_factors=2,         # Number of factors
    factor_order=2       # AR order of factors
)
results = model.fit(disp=False)

# STEP 3: Extract factors
factors = results.factors.filtered

# STEP 4: Get factor loadings
loadings = results.params['loading']

# STEP 5: Make predictions
forecast = results.forecast(steps=12)

print("✓ Done! Your factors are ready.")

## What's Next?

You just built your first Dynamic Factor Model! Here's what to explore next:

1. **`01_your_first_forecast.ipynb`** - Use DFM to forecast real economic data
2. **`02_multivariate_example.ipynb`** - Handle 10+ series with missing data
3. **`../concepts/visual_guides/factor_interpretation.md`** - Learn what factors mean
4. **`../templates/dfm_pipeline_template.py`** - Production-ready template

---

**Key Takeaway:** DFMs compress many correlated time series into a few interpretable factors, making forecasting and analysis much simpler.